# Auto-RAN Blueprint

## Prerequisites & Environment setup

Install **docker** and **docker compose** if they are missing.

In [ ]:
!command -v docker >/dev/null 2>&1 || (curl -fsSL https://get.docker.com -o get-docker.sh && sudo sh get-docker.sh)
!docker compose version >/dev/null 2>&1 || sudo apt-get install -y docker-compose-plugin

Install Python dependencies.

In [ ]:
%pip install requests

Agents need some environment variables to work properly. The following script helps you generating a proper `.env` file for each agent.

Please, paste your Nvidia API Key in the following block of code before running it.

In [ ]:
api_key = 

Run the following block to setup required environment variables for the agents.

**Note**: the script sets both the `API_KEY` and `NVIDIA_API_KEY` environment variables because the toolkits used for development (BAT-ADK and NAT) expect the `API_KEY` variable to be set, while the Nvidia model expects the `NVIDIA_API_KEY` variable to be set.

In [ ]:
from pathlib import Path

agents = {
    "config_planner": 9201,
    "monitoring": 9202,
    "validation": 9203,
}

template = """URL=http://localhost/
PORT={port}
MODEL=nvidia:meta/llama-3.1-70b-instruct
API_KEY={api_key}
NVIDIA_API_KEY={api_key}
"""

for agent, port in agents.items():
    path = Path(f"./agents/{agent}")
    path.mkdir(parents=True, exist_ok=True)

    env_file = path / ".env"
    if not env_file.exists():  # only write if it doesn't exist
        env_file.write_text(template.format(port=port, api_key=api_key))
        print(f"✅ Created {env_file}")
    else:
        print(f"⚠️ {env_file} already exists, skipping.")

print("✅ Done")

In [ ]:
# TODO: add setup api call
# Needed to optimize resources utilization on BubbleRAN server

## Build the agents

Run `docker compose build` to build the agents container images.

In [ ]:
!docker compose build

## Run the agents

Start the containers in **detached** mode (`-d`).

In [ ]:
!docker compose up -d

Check which containers are active with `docker ps`.

In [ ]:
!docker ps

## Use the agents

The following block of code defines a function which sends requests to the **Config Planner**. Please, run it to be able to easily send requests to the agent.

You may need to change the `context_id` parameter when you call this function for multiple times.

In [ ]:
import requests
import json

def stream_config_planner(context_id: str, user_text: str):
    headers = {
        "Content-Type": "application/json",
    }
    payload = {
        "jsonrpc": "2.0",
        "id": 1,
        "method": "message/stream",
        "params": {
            "configuration": {"accepted_output_modes": ["text"]},
            "message": {
                "context_id": context_id,
                "message_id": "1",
                "role": "user",
                "parts": [{"type": "text", "text": user_text}]
            }
        }
    }
    last_text = None
    config_planner_url = "http://localhost:9201"

    with requests.post(config_planner_url, headers=headers, json=payload, stream=True) as response:
        if response.status_code != 200:
            print("Request failed:", response.status_code, response.text)
            return

        for line in response.iter_lines():
            if not line:
                continue
            try:
                text = line.decode("utf-8")
                if text.startswith("data:"):
                    text = text[len("data:"):].strip()
                data = json.loads(text)
                result = data.get("result", {})

                if "status" in result:
                    message = result["status"].get("message", {})
                    for part in message.get("parts", []):
                        if part.get("kind") == "text" and part["text"] != last_text:
                            print("> " + part["text"])
                            last_text = part["text"]

                # Artifact updates (final output)
                elif "artifact" in result:
                    for part in result["artifact"].get("parts", []):
                        if part.get("kind") == "text" and part["text"] != last_text:
                            print("\n--- FINAL ANSWER ---\n")
                            print(part["text"])
                            last_text = part["text"]

            except json.JSONDecodeError:
                continue

Send a monitoring request to the **Config Planner**.

In [ ]:
stream_config_planner(
    context_id="1",
    user_text="What are the current P0 nominal and uplink throughput of the 'colsstb01' network?"
)

Send an optimization request to the **Config Planner**.

In [ ]:
stream_config_planner(
    context_id="1",
    user_text="Please optimize the P0 nominal of the 'colsstb01' network to improve uplink throughput"
)

## Stop the containers and clean-up the resources

In [ ]:
!docker compose down

In [ ]:
# TODO: add cleanup api call
# Needed to optimize resources utilization on BubbleRAN server